In [ ]:
import pandas as pd

# Load the CSV file into a DataFrame
df = pd.read_csv('C:\\Users\\Travail\\OneDrive\\Desktop\\FUFA\\League Matches\\UPL\\Kcca Fc Matches\\Kcca Fc_league_matches_all.csv') 

# Display the first few rows of the DataFrame (for debugging/inspection purposes)
df.head()

# Standardize column names: strip whitespace, replace spaces with underscores, and convert to lowercase
df.columns = df.columns.str.strip().str.replace(' ', '_').str.lower()

# Convert the 'date' column to datetime format, handling Excel-style date origin and coercing errors
df['date'] = pd.to_datetime(df['date'], origin='1899-12-30', unit='D', errors='coerce')

# Drop columns with names matching a specific regex pattern (e.g., time in HR load zones)
df = df[df.columns.drop(df.filter(regex=r'time_in_hr_load_zone_.*%_-_.*%_max_hr').columns)]

# Define a list of columns to remove and drop them from the DataFrame, ignoring errors if they don't exist
columns_to_remove = ['split_start_time', 'split_end_time', 'hr_load', 'hr_max_(bpm)', 'time_in_red_zone_(min)', 'league', 'tags']
df = df.drop(columns=columns_to_remove, errors='ignore')

# Strip whitespace and convert string values to lowercase for all columns
df = df.map(lambda x: x.strip().lower() if isinstance(x, str) else x)

# Remove rows where the 'source_file' column matches a specific value
df = df[df['source_file'] != 'kcca fc_league_matches_all.csv']

# Remove rows where the 'split_name' column equals 'all'
df = df[df['split_name'] != 'all']

# Split the 'session_title' column into multiple columns and assign them to new columns
df[['match_day', 'home_team', 'away_team', 'location', 'league', 'result']] = df['session_title'].str.split('-', expand=True)

# Split the 'player_name' column into multiple columns and assign them to new columns
df[['player_nam', 'team', 'player_position']] = df['player_name'].str.split('-', expand=True)

# Drop additional unnecessary columns
columns_to_remove = ['league', 'session_title', 'source_file', 'player_name']
df = df.drop(columns=columns_to_remove, errors='ignore')

# Extract and clean the 'team' and 'player_position_' columns from the 'team' column
df[['team', 'player_position_']] = df['team'].str.extract(r'^(kcca_)?(.*)$')
df['player_position_'] = df['player_position_'].fillna(df['team'])

# Update 'player_position' based on cleaned 'player_position_' values
df.loc[df['player_position_'].isin(['mf', 'df', 'fw']), 'player_position'] = df['player_position_']

# Drop intermediate columns used for cleaning
df = df.drop(columns=['player_position_', 'team'], errors='ignore')

# Standardize player positions (e.g., replace 'amc' with 'am')
df['player_position'] = df['player_position'].str.replace('amc', 'am')

# Clean and convert the 'match_day' column to integers
df['match_day'] = df['match_day'].str.replace('md', '').astype(int)

# Convert 'duration' from seconds to minutes and round to two decimal places
df['duration'] = round(df['duration'] // 60, 2)

# Drop the 'index' column if it exists
df = df.drop(columns=['index'], errors='ignore')

# Rename columns for clarity
df = df.rename(columns={'player_nam': 'player_name', 'duration': 'duration(min)'})

# Separate rows where players were on the bench (duration = 0)
on_bench_df = df[df['duration(min)'] == 0]

# Keep only rows where players had non-zero duration
df = df[df['duration(min)'] != 0]

# Map specific player positions to general positions
df['general_position'] = df['player_position'].map({
    'gk': 'goalkeeper',
    'df': 'defender',
    'mf': 'midfielder',
    'am': 'midfielder',
    'fw': 'forward',
    'rb': 'defender',
    'cb': 'defender',
    'lb': 'defender',
    'rw': 'forward',
    'lw': 'forward',
    'cm': 'midfielder',
    'dm': 'midfielder',
})

# Sort the DataFrames by the 'date' column in descending order and reset the index
df = df.sort_values(by='date', ascending=False).reset_index()
on_bench_df = on_bench_df.sort_values(by='date', ascending=False).reset_index()

# df.to_csv('C:\\Users\\Travail\\OneDrive\\Desktop\\FUFA\\League Matches\\UPL\\Kcca Fc Matches\\Kcca Fc_league_matches_cleaned.csv', index=False)
# on_bench_df.to_csv('C:\\Users\\Travail\\OneDrive\\Desktop\\FUFA\\League Matches\\UPL\\Kcca Fc Matches\\Kcca Fc_league_matches_benched.csv', index=False)